In [1]:
# Importation des libraries nécessaires
import pandas as pd

In [ ]:
# importation de la base de données
df = pd.read_csv('./data/hvstat_africa_data_v1.0.csv', index_col=0)

df.head()

In [ ]:
def clean_data(df):
    # Filter West Africa countries
    west_africa_countries = ['Benin', 'Burkina Faso', 'Cape Verde', 'Côte d\'Ivoire', 'Gambia', 'Ghana', 'Guinea', 'Guinea-Bissau', 'Liberia', 'Mali', 'Niger', 'Nigeria', 'Senegal', 'Sierra Leone', 'Togo', 'Mauritania']
    df = df[df['country'].isin(west_africa_countries)]
    return df

df_clean = clean_data(df.copy())
df_clean.head()

In [5]:
df_clean[['country', 'admin_2']].drop_duplicates().sort_values(by=['country', 'admin_2']).head(20)

,country,admin_2
fnid,,
BJ2015A20307,Benin,Abomey
BJ2015A20602,Benin,Abomey-Calavi
BJ2015A20801,Benin,Adja-Ouèrè
BJ2015A20206,Benin,Adjarra
BJ2015A20207,Benin,Adjohoun
BJ2015A20308,Benin,Agbangnizoun
BJ2015A20208,Benin,Aguégués
BJ2015A20209,Benin,Akpro-Missérété
BJ2015A20603,Benin,Allada


In [ ]:
df2 = df_clean[['country', 'admin_2']].drop_duplicates().sort_values(by=['country', 'admin_2'])

df2.head()

In [8]:
# Valeur différente de "none" dans la colonne "admin_2"
df3 = df2[df2['admin_2'] != 'none']
df3.head()

,country,admin_2
fnid,,
BJ2015A20307,Benin,Abomey
BJ2015A20602,Benin,Abomey-Calavi
BJ2015A20801,Benin,Adja-Ouèrè
BJ2015A20206,Benin,Adjarra
BJ2015A20207,Benin,Adjohoun


In [10]:
# Géocodage des unités admin_2 avec leur pays pour éviter les homonymes
import time
from pathlib import Path

import requests

cache_path = Path('./data/admin2_coordinates.csv')
if cache_path.exists():
    cache_df = pd.read_csv(cache_path)
else:
    cache_df = pd.DataFrame(columns=['country', 'admin_2', 'latitude', 'longitude'])

coordinates = {
    (row.country, row.admin_2): (row.latitude, row.longitude)
    for row in cache_df.itertuples(index=False)
}

places = (
    df3[['country', 'admin_2']]
    .dropna()
    .drop_duplicates()
    .query("admin_2.str.lower() != 'none'", engine='python')
    .reset_index(drop=True)
    .itertuples(index=False)
 )
for number, place in enumerate(places, start=1):
    key = (place.country, place.admin_2)
    if key not in coordinates:
        response = requests.get(
            'https://nominatim.openstreetmap.org/search',
            params={
                'q': f'{place.admin_2}, {place.country}',
                'format': 'jsonv2',
                'limit': 1,
            },
            headers={
                'User-Agent': 'Projet3DataAgroMeteorologique/1.0 (educational project)'
            },
            timeout=20,
        )
        response.raise_for_status()
        results = response.json()
        if results:
            coordinates[key] = (float(results[0]['lat']), float(results[0]['lon']))
        else:
            coordinates[key] = (float('nan'), float('nan'))

        # Nominatim asks clients to stay at or below one request per second.
        time.sleep(1.1)
        if number % 10 == 0:
            pd.DataFrame(
                [(*key, lat, lon) for key, (lat, lon) in coordinates.items()],
                columns=['country', 'admin_2', 'latitude', 'longitude'],
            ).to_csv(cache_path, index=False)

pd.DataFrame(
    [(*key, lat, lon) for key, (lat, lon) in coordinates.items()],
    columns=['country', 'admin_2', 'latitude', 'longitude'],
 ).to_csv(cache_path, index=False)

df3_geo = df3.merge(
    pd.DataFrame(
        [(*key, lat, lon) for key, (lat, lon) in coordinates.items()],
        columns=['country', 'admin_2', 'latitude', 'longitude'],
    ),
    on=['country', 'admin_2'],
    how='left',
)
df3_geo.head()

,country,admin_2,latitude,longitude
0,Benin,Abomey,7.182001,1.993632
1,Benin,Abomey-Calavi,6.453864,2.354245
2,Benin,Adja-Ouèrè,7.011825,2.598304
3,Benin,Adjarra,6.493028,2.693338
4,Benin,Adjohoun,6.698138,2.516411


In [ ]:
# Liste des lieux avec leurs coordonnées
admin2 = [
    {
        'nom': row.admin_2,
        'pays': row.country,
        'lat': row.latitude,
        'lon': row.longitude,
    }
    for row in df3_geo.dropna(subset=['latitude', 'longitude']).itertuples(index=False)
 ]

admin2[:5]